# SC4001 Phase 2 baseline, DAVIS grid

Accelerator must be **GPU (T4 x2 or P100)**. This code is plain PyTorch CUDA, so on a
TPU node `torch.cuda.is_available()` is false and `src/train.py` refuses to run by design.

Code and processed tables come from the attached dataset `isaacwjk21/sc4001-project`.
No internet and no installs are needed: torch, numpy, pandas and scipy are preinstalled.

Grid: DAVIS `split_warm` and `split_cold_target`, seeds 4001/4002/4003, six runs.
The warm-minus-cold-target gap is the number Phase 3 (DANN) tries to shrink.
Outputs go to `/kaggle/working/outputs` and survive only if this version is saved.

In [ ]:
import json, os, shutil, subprocess, sys, time
from pathlib import Path
import torch

if not torch.cuda.is_available():
    raise SystemExit("No CUDA device. Set the accelerator to GPU T4 x2 or P100, not TPU or None.")
props = torch.cuda.get_device_properties(0)
print("torch", torch.__version__, "| device", props.name,
      "|", round(props.total_memory / 1e9, 1), "GB | count", torch.cuda.device_count())

In [ ]:
# Show what is actually mounted, so a layout surprise is visible rather than guessed at.
inputs = sorted(Path("/kaggle/input").rglob("*"))
print(len(inputs), "mounted paths; first 25:")
for path in inputs[:25]:
    print("   ", path)

# Locate the project root: the directory holding both src/train.py and data/processed/vocab.json.
# Kaggle strips the archive's top-level folder, so do not assume a "Project" path segment.
candidates = [p.parents[1] for p in Path("/kaggle/input").rglob("src/train.py")]
roots = [p for p in candidates if (p / "data" / "processed" / "vocab.json").exists()]
if not roots:
    raise SystemExit(
        "No project root found. Looked for src/train.py beside data/processed/vocab.json. "
        "Candidates with src/train.py but no vocab.json: " + str(candidates)
    )
source = sorted(roots)[0]

project = Path("/kaggle/working/sc4001/Project")
shutil.copytree(source, project, dirs_exist_ok=True)
env = dict(os.environ, SC4001_OUTPUT_ROOT="/kaggle/working/outputs", PYTHONUNBUFFERED="1")
print("source :", source)
print("project:", project)
print("tables :", sorted(p.name for p in (project / "data" / "processed").iterdir()))

In [ ]:
# Smoke run first: proves the pipeline end to end on this GPU before spending quota.
smoke = subprocess.run(
    [sys.executable, "src/train.py", "--dataset", "davis", "--split", "split_warm", "--smoke"],
    cwd=project, env=env, check=False,
)
if smoke.returncode != 0:
    raise SystemExit("Smoke run failed with code " + str(smoke.returncode) + "; not starting the grid.")
shutil.rmtree("/kaggle/working/outputs/phase2/davis/split_warm/seed4001-smoke", ignore_errors=True)
print("smoke OK")

In [ ]:
GRID = [("davis", split, seed)
        for split in ("split_warm", "split_cold_target")
        for seed in (4001, 4002, 4003)]
started = time.time()
status = []
for dataset, split, seed in GRID:
    t0 = time.time()
    print()
    print("===== " + dataset + " " + split + " seed " + str(seed) + " =====", flush=True)
    run = subprocess.run(
        [sys.executable, "src/train.py", "--dataset", dataset, "--split", split,
         "--seed", str(seed), "--resume"],
        cwd=project, env=env, check=False,
    )
    elapsed = time.time() - t0
    status.append((dataset, split, seed, run.returncode, elapsed))
    print("----- rc=" + str(run.returncode) + " wall=" + format(elapsed / 60, ".1f") + " min -----", flush=True)

print()
print("Grid wall time", format((time.time() - started) / 60, ".1f"), "min")
for ds, sp, sd, rc, el in status:
    print(f"{ds} {sp:18s} seed{sd}  rc={rc}  {el / 60:6.1f} min")
failed = [row for row in status if row[3] != 0]
print("FAILED RUNS:", failed if failed else "none")

In [ ]:
# Print every metrics.json so the numbers live in the saved notebook even if files are lost.
found = sorted(Path("/kaggle/working/outputs").glob("**/metrics.json"))
print(len(found), "metrics files")
rows = []
for path in found:
    payload = json.loads(path.read_text())
    print("=" * 70)
    print(path)
    print(json.dumps(payload, indent=2))
    model = payload["test"]["all"]
    floor = payload["training_mean_baseline"]["test"]["all"]
    rows.append((payload["split"], payload["seed"], model["mse"], model["ci"],
                 floor["mse"], floor["ci"], payload["best_epoch"]))

print("=" * 86)
print(f"{'split':20s} {'seed':>5s} {'mse':>9s} {'ci':>8s} {'mean_mse':>9s} {'mean_ci':>8s} {'best_ep':>8s}")
for split, seed, m, c, bm, bc, ep in sorted(rows):
    bc_text = "     n/a" if bc is None else format(bc, ">8.4f")
    print(f"{split:20s} {seed:>5d} {m:>9.4f} {c:>8.4f} {bm:>9.4f} {bc_text} {ep:>8d}")

# Headline number: warm minus cold-target, paired by seed.
by_key = {(r[0], r[1]): (r[2], r[3]) for r in rows}
warm_seeds = {k[1] for k in by_key if k[0] == "split_warm"}
cold_seeds = {k[1] for k in by_key if k[0] == "split_cold_target"}
shared = sorted(warm_seeds & cold_seeds)
if shared:
    print()
    print("Cold-target minus warm, paired by seed:")
    for sd in shared:
        wm, wc = by_key[("split_warm", sd)]
        cm, cc = by_key[("split_cold_target", sd)]
        print(f"  seed {sd}: dMSE {cm - wm:+.4f}   dCI {cc - wc:+.4f}")

## Run record

Executed headlessly on Kaggle as `isaacwjk21/notebook8c0fa53c58` on a Tesla T4
(`torch 2.11.0+cu130`), status COMPLETE. Code and processed tables came from the
dataset `isaacwjk21/sc4001-project`; metrics and checkpoints were downloaded with
`kaggle kernels output` into `outputs/phase2/`.

The stdout below is the whole run's captured log, attached to the final cell.
Kaggle's API does not say which cell produced which line, so it is not split up.

In [ ]:
# Captured stdout of the Kaggle run (see the markdown cell above).


torch 2.11.0+cu130 | device Tesla T4 | 15.6 GB | count 2
21 mounted paths; first 25:
    /kaggle/input/sc4001-project
    /kaggle/input/sc4001-project/data
    /kaggle/input/sc4001-project/data/processed
    /kaggle/input/sc4001-project/data/processed/davis_ligands.csv
    /kaggle/input/sc4001-project/data/processed/davis_mutants.csv
    /kaggle/input/sc4001-project/data/processed/davis_pairs.csv
    /kaggle/input/sc4001-project/data/processed/davis_targets.csv
    /kaggle/input/sc4001-project/data/processed/davis_tokens.npz
    /kaggle/input/sc4001-project/data/processed/kiba_ligands.csv
    /kaggle/input/sc4001-project/data/processed/kiba_pairs.csv
    /kaggle/input/sc4001-project/data/processed/kiba_targets.csv
    /kaggle/input/sc4001-project/data/processed/kiba_tokens.npz
    /kaggle/input/sc4001-project/data/processed/vocab.json
    /kaggle/input/sc4001-project/src
    /kaggle/input/sc4001-project/src/__init__.py
    /kaggle/input/sc4001-project/src/config.py
    /kaggle/input/sc